In [25]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder


train = pd.read_csv("train_data.csv")
pd.set_option('display.max_columns', None)
train.head(10)


,id,Birth_Date,Weight,Height,Urban_Rural,Occupation,Insurance_Type,Family_History,Cancer_Type,Stage_at_Diagnosis,Diagnosis_Date,Symptoms,Tumor_Size,Surgery_Date,Chemotherapy_Drugs,Radiation_Sessions,Immunotherapy,Targeted_Therapy,Recurrence_Status,Smoking_History,Alcohol_Use,label
0,1,1994-07-01,64.9,155.0cm,Urban,Unemployed,UEBMI,No,Breast,II,2020-02-10,"Cough, Weight Loss",8.0,2024-10-19,"Paclitaxel,Docetaxel,Doxorubicin",16,No,Yes,NO,Never,Regular,1
1,2,1992-07-16,61.4,171.0cm,Urban,Factory Worker,UEBMI,Yes,Breast,I,2014-08-17,Blood in Stool,10.0,2021-02-28,"Cyclophosphamide,Paclitaxel,Doxorubicin,Docetaxel",10,No,No,Yes,Former,Regular,1
2,3,1948-06-23,60.7,170.0cm,Rural,Unemployed,NRCMS,No,Stomach,IV,2014-09-25,"Nausea, Vomiting",13.0,2022-09-25,"Fluorouracil,Cisplatin",21,Yes,No,NO,Former,Never,0
3,4,1954-11-26,70.2,171.0cm,Urban,Farmer,URBMI,Yes,Cervical,IV,2021-01-04,"Nausea, Vomiting",3.0,2024-09-13,Cisplatin,10,No,Yes,NO,Never,Regular,1
4,5,1979-07-08,100.3,186.0cm,Rural,Office Worker,Self-pay,Yes,Lung,II,2019-07-26,"Cough, Weight Loss",12.0,2023-12-08,"Gemcitabine,Carboplatin",6,Yes,No,Yes,Former,Never,0
5,6,1958-08-09,49.6,175.0cm,Rural,Retired,Self-pay,No,Breast,II,2016-02-10,"Nausea, Vomiting",8.7,2022-02-24,"Gemcitabine, Carboplatin",13,Yes,No,Yes,Never,Never,1
6,7,1991-11-09,109.5,185.0cm,Rural,Unemployed,Self-pay,Yes,Esophageal,IV,2022-08-04,"Lump, Swelling",4.0,2023-07-28,"Cisplatin,Fluorouracil",6,No,Yes,NO,Former,Regular,0
7,8,1954-07-27,65.7,186.0cm,Rural,Factory Worker,Self-pay,No,Cervical,I,2010-09-27,"Nausea, Vomiting",4.0,2017-01-10,"Paclitaxel,Cisplatin",4,No,Yes,NO,Former,Occasional,1
8,9,1984-07-03,50.6,159.0cm,Urban,Factory Worker,URBMI,Yes,Lung,IV,2020-03-31,"Cough, Weight Loss",14.0,2023-03-10,"Carboplatin,Gemcitabine",26,No,No,Yes,Current,Never,0
9,10,1962-06-26,88.8,164.0cm,Urban,Unemployed,URBMI,Yes,Colorectal,I,2018-07-31,"Cough, Weight Loss",6.0,2013-12-19,"Fluorouracil,Irinotecan,Oxaliplatin,Leucovorin",10,Yes,Yes,NO,Never,Occasional,0


In [26]:
le = LabelEncoder()
train['Family_History'] = le.fit_transform(train['Family_History'])
train['Immunotherapy'] = le.fit_transform(train['Immunotherapy'])
train['Targeted_Therapy'] = le.fit_transform(train['Targeted_Therapy'])
train['Recurrence_Status'] = le.fit_transform(train['Recurrence_Status'])


In [27]:
train = pd.get_dummies(train, columns=['Urban_Rural','Occupation', 'Insurance_Type' ,'Cancer_Type' ,'Smoking_History', 'Alcohol_Use'], drop_first=False)

roman_to_int = {
    'I': 1,
    'II': 2,
    'III': 3,
    'IV': 4,
}

train['Stage_at_Diagnosis'] = train['Stage_at_Diagnosis'].map(lambda x: roman_to_int.get(x, 0))


In [28]:
from sklearn.preprocessing import MultiLabelBinarizer

train['Drug_List'] = train['Chemotherapy_Drugs'].fillna('').apply(
    lambda x: [drug.strip() for drug in str(x).split(',')] if x else []
)

mlb = MultiLabelBinarizer()
drug_matrix = mlb.fit_transform(train['Drug_List'])

drug_df = pd.DataFrame(drug_matrix, columns=[f"Drug_{drug}" for drug in mlb.classes_])
train = pd.concat([train.reset_index(drop=True), drug_df.reset_index(drop=True)], axis=1)

In [29]:

train['Symptom_List'] = train['Symptoms'].fillna('').apply(
    lambda x: [sym.strip() for sym in str(x).split(',')] if x else []
)

mlb_symptoms = MultiLabelBinarizer()
symptom_matrix = mlb_symptoms.fit_transform(train['Symptom_List'])

symptom_df = pd.DataFrame(symptom_matrix, columns=[f"Symptom_{s.replace(' ', '_')}" for s in mlb_symptoms.classes_])
train = pd.concat([train.reset_index(drop=True), symptom_df.reset_index(drop=True)], axis=1)

In [30]:
train['Birth_Date'] = pd.to_datetime(train['Birth_Date'],errors='coerce')
train['Diagnosis_Date'] = pd.to_datetime(train['Diagnosis_Date'],errors='coerce')
train['Surgery_Date'] = pd.to_datetime(train['Surgery_Date'], errors='coerce')
train['Age_at_Diagnosis'] = train['Diagnosis_Date'].dt.year - train['Birth_Date'].dt.year
train['Age_at_Surgery'] = train['Surgery_Date'].dt.year - train['Birth_Date'].dt.year


In [31]:
train['Height'] = train['Height'].str.replace('cm', '', regex=False).astype(float)
train['Height'] = pd.to_numeric(train['Height'], errors='coerce')
train['Weight'] = pd.to_numeric(train['Weight'], errors='coerce')
train['BMI'] = (train['Weight'] / ((train['Height']/100) ** 2)).round(2)
train.head()


,id,Birth_Date,Weight,Height,Family_History,Stage_at_Diagnosis,Diagnosis_Date,Symptoms,Tumor_Size,Surgery_Date,Chemotherapy_Drugs,Radiation_Sessions,Immunotherapy,Targeted_Therapy,Recurrence_Status,label,Urban_Rural_Rural,Urban_Rural_Urban,Occupation_Factory Worker,Occupation_Farmer,Occupation_Office Worker,Occupation_Retired,Occupation_Unemployed,Insurance_Type_NRCMS,Insurance_Type_Self-pay,Insurance_Type_UEBMI,Insurance_Type_URBMI,Cancer_Type_Breast,Cancer_Type_Cervical,Cancer_Type_Colorectal,Cancer_Type_Esophageal,Cancer_Type_Liver,Cancer_Type_Lung,Cancer_Type_Stomach,Smoking_History_Current,Smoking_History_Former,Smoking_History_Never,Alcohol_Use_Never,Alcohol_Use_Occasional,Alcohol_Use_Regular,Drug_List,Drug_Carboplatin,Drug_Cisplatin,Drug_Cyclophosphamide,Drug_Docetaxel,Drug_Doxorubicin,Drug_Fluorouracil,Drug_Gemcitabine,Drug_Irinotecan,Drug_Leucovorin,Drug_Oxaliplatin,Drug_Paclitaxel,Drug_Sorafenib,Symptom_List,Symptom_Blood_in_Stool,Symptom_Cough,Symptom_Fatigue,Symptom_Lump,Symptom_Nausea,Symptom_Pain,Symptom_Swelling,Symptom_Vomiting,Symptom_Weight_Loss,Age_at_Diagnosis,Age_at_Surgery,BMI
0,1,1994-07-01,64.9,155.0,0,2,2020-02-10,"Cough, Weight Loss",8.0,2024-10-19,"Paclitaxel,Docetaxel,Doxorubicin",16,0,1,0,1,False,True,False,False,False,False,True,False,False,True,False,True,False,False,False,False,False,False,False,False,True,False,False,True,"[Paclitaxel, Docetaxel, Doxorubicin]",0,0,0,1,1,0,0,0,0,0,1,0,"[Cough, Weight Loss]",0,1,0,0,0,0,0,0,1,26.0,30.0,27.01
1,2,1992-07-16,61.4,171.0,1,1,2014-08-17,Blood in Stool,10.0,2021-02-28,"Cyclophosphamide,Paclitaxel,Doxorubicin,Docetaxel",10,0,0,1,1,False,True,True,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,True,False,False,False,True,"[Cyclophosphamide, Paclitaxel, Doxorubicin, Do...",0,0,1,1,1,0,0,0,0,0,1,0,[Blood in Stool],1,0,0,0,0,0,0,0,0,22.0,29.0,21.00
2,3,1948-06-23,60.7,170.0,0,4,2014-09-25,"Nausea, Vomiting",13.0,2022-09-25,"Fluorouracil,Cisplatin",21,1,0,0,0,True,False,False,False,False,False,True,True,False,False,False,False,False,False,False,False,False,True,False,True,False,True,False,False,"[Fluorouracil, Cisplatin]",0,1,0,0,0,1,0,0,0,0,0,0,"[Nausea, Vomiting]",0,0,0,0,1,0,0,1,0,66.0,74.0,21.00
3,4,1954-11-26,70.2,171.0,1,4,2021-01-04,"Nausea, Vomiting",3.0,2024-09-13,Cisplatin,10,0,1,0,1,False,True,False,True,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,True,False,False,True,[Cisplatin],0,1,0,0,0,0,0,0,0,0,0,0,"[Nausea, Vomiting]",0,0,0,0,1,0,0,1,0,67.0,70.0,24.01
4,5,1979-07-08,100.3,186.0,1,2,2019-07-26,"Cough, Weight Loss",12.0,2023-12-08,"Gemcitabine,Carboplatin",6,1,0,1,0,True,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,True,False,False,True,False,True,False,False,"[Gemcitabine, Carboplatin]",1,0,0,0,0,0,1,0,0,0,0,0,"[Cough, Weight Loss]",0,1,0,0,0,0,0,0,1,40.0,44.0,28.99


In [32]:

train.drop(columns=['Birth_Date'], inplace=True)
train.drop(columns=['Diagnosis_Date'], inplace=True)
train.drop(columns=['Surgery_Date'], inplace=True)
train.drop(columns=['Symptoms', 'Symptom_List'], inplace=True)
train.drop(columns=['Chemotherapy_Drugs', 'Drug_List'], inplace=True)
train = train.map(lambda x: 1 if x is True else (0 if x is False else x))



In [33]:
train.head(10)

,id,Weight,Height,Family_History,Stage_at_Diagnosis,Tumor_Size,Radiation_Sessions,Immunotherapy,Targeted_Therapy,Recurrence_Status,label,Urban_Rural_Rural,Urban_Rural_Urban,Occupation_Factory Worker,Occupation_Farmer,Occupation_Office Worker,Occupation_Retired,Occupation_Unemployed,Insurance_Type_NRCMS,Insurance_Type_Self-pay,Insurance_Type_UEBMI,Insurance_Type_URBMI,Cancer_Type_Breast,Cancer_Type_Cervical,Cancer_Type_Colorectal,Cancer_Type_Esophageal,Cancer_Type_Liver,Cancer_Type_Lung,Cancer_Type_Stomach,Smoking_History_Current,Smoking_History_Former,Smoking_History_Never,Alcohol_Use_Never,Alcohol_Use_Occasional,Alcohol_Use_Regular,Drug_Carboplatin,Drug_Cisplatin,Drug_Cyclophosphamide,Drug_Docetaxel,Drug_Doxorubicin,Drug_Fluorouracil,Drug_Gemcitabine,Drug_Irinotecan,Drug_Leucovorin,Drug_Oxaliplatin,Drug_Paclitaxel,Drug_Sorafenib,Symptom_Blood_in_Stool,Symptom_Cough,Symptom_Fatigue,Symptom_Lump,Symptom_Nausea,Symptom_Pain,Symptom_Swelling,Symptom_Vomiting,Symptom_Weight_Loss,Age_at_Diagnosis,Age_at_Surgery,BMI
0,1,64.9,155.0,0,2,8.0,16,0,1,0,1,0,1,0,0,0,0,1,0,0,1,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,1,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,26.0,30.0,27.01
1,2,61.4,171.0,1,1,10.0,10,0,0,1,1,0,1,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,1,1,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,22.0,29.0,21.00
2,3,60.7,170.0,0,4,13.0,21,1,0,0,0,1,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,1,0,1,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,66.0,74.0,21.00
3,4,70.2,171.0,1,4,3.0,10,0,1,0,1,0,1,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,67.0,70.0,24.01
4,5,100.3,186.0,1,2,12.0,6,1,0,1,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,40.0,44.0,28.99
5,6,49.6,175.0,0,2,8.7,13,1,0,1,1,1,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,58.0,64.0,16.20
6,7,109.5,185.0,1,4,4.0,6,0,1,0,0,1,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,31.0,32.0,31.99
7,8,65.7,186.0,0,1,4.0,4,0,1,0,1,1,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,56.0,63.0,18.99
8,9,50.6,159.0,1,4,14.0,26,0,0,1,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,36.0,39.0,20.02
9,10,88.8,164.0,1,1,6.0,10,1,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,1,1,1,0,0,0,1,0,0,0,0,0,0,1,56.0,51.0,33.02


In [34]:
from sklearn.model_selection import train_test_split

X = train.drop(columns=['label'])
y = train['label']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)




RandomForestClassifier(random_state=42)

In [35]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

y_val_pred = model.predict(X_val)

print("🔍 Model Evaluation on Validation Set:")
print("Accuracy:", accuracy_score(y_val, y_val_pred))
print("Precision:", precision_score(y_val, y_val_pred))
print("Recall:", recall_score(y_val, y_val_pred))
print("F1 Score:", f1_score(y_val, y_val_pred))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_val, y_val_pred)
print(cm)



🔍 Model Evaluation on Validation Set:
Accuracy: 0.7962228517469311
Precision: 0.7994384275972723
Recall: 0.7748833592534993
F1 Score: 0.7869693978282329

Confusion Matrix:
[[2223  500]
 [ 579 1993]]


In [36]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer

test = pd.read_csv("test_data.csv")
test_ids = test['id'] 

le = LabelEncoder()
for col in ['Family_History', 'Immunotherapy', 'Targeted_Therapy', 'Recurrence_Status']:
    test[col] = le.fit_transform(test[col])

test = pd.get_dummies(test, columns=['Urban_Rural','Occupation', 'Insurance_Type', 'Cancer_Type',
                                     'Smoking_History', 'Alcohol_Use'], drop_first=False)

roman_to_int = {'I': 1, 'II': 2, 'III': 3, 'IV': 4}
test['Stage_at_Diagnosis'] = test['Stage_at_Diagnosis'].map(lambda x: roman_to_int.get(x, 0))

test['Drug_List'] = test['Chemotherapy_Drugs'].fillna('').apply(
    lambda x: [drug.strip() for drug in str(x).split(',')] if x else []
)

mlb_drugs = MultiLabelBinarizer(classes=[
    'Carboplatin','Cisplatin','Cyclophosphamide','Docetaxel','Doxorubicin',
    'Fluorouracil','Gemcitabine','Irinotecan','Leucovorin','Oxaliplatin',
    'Paclitaxel','Sorafenib'
])
drug_matrix = mlb_drugs.fit_transform(test['Drug_List'])
drug_df = pd.DataFrame(drug_matrix, columns=[f"Drug_{drug}" for drug in mlb_drugs.classes_])
test = pd.concat([test.reset_index(drop=True), drug_df.reset_index(drop=True)], axis=1)

test['Symptom_List'] = test['Symptoms'].fillna('').apply(
    lambda x: [s.strip() for s in str(x).split(',')] if x else []
)

mlb_sym = MultiLabelBinarizer(classes=[
    'Blood in Stool','Cough','Fatigue','Lump','Nausea',
    'Pain','Swelling','Vomiting','Weight Loss'
])
sym_matrix = mlb_sym.fit_transform(test['Symptom_List'])
sym_df = pd.DataFrame(sym_matrix, columns=[f"Symptom_{s.replace(' ', '_')}" for s in mlb_sym.classes_])
test = pd.concat([test.reset_index(drop=True), sym_df.reset_index(drop=True)], axis=1)

test['Birth_Date'] = pd.to_datetime(test['Birth_Date'], errors='coerce')
test['Diagnosis_Date'] = pd.to_datetime(test['Diagnosis_Date'], errors='coerce')
test['Surgery_Date'] = pd.to_datetime(test['Surgery_Date'], errors='coerce')
test['Age_at_Diagnosis'] = test['Diagnosis_Date'].dt.year - test['Birth_Date'].dt.year
test['Age_at_Surgery'] = test['Surgery_Date'].dt.year - test['Birth_Date'].dt.year

test['Height'] = test['Height'].str.replace('cm', '', regex=False).astype(float)
test['Weight'] = pd.to_numeric(test['Weight'], errors='coerce')
test['BMI'] = (test['Weight'] / ((test['Height']/100)**2)).round(2)

test.drop(columns=[
    'Birth_Date','Diagnosis_Date','Surgery_Date',
    'Symptoms','Symptom_List','Chemotherapy_Drugs','Drug_List'
], inplace=True)

test = test.map(lambda x: 1 if x is True else (0 if x is False else x))

final_columns = [col for col in X_train.columns]  
for col in final_columns:
    if col not in test.columns:
        test[col] = 0  

test = test[final_columns]

y_pred = model.predict(test)

submission = pd.DataFrame({
    'id': test_ids,
    'label': y_pred
})
submission.to_csv("submission.csv", index=False)
